In [1]:
#!unzip military_vehicles_dataset.zip

In [2]:
from pathlib import Path

In [3]:
data_path = Path("")
dataset_path = data_path / "military_vehicles_dataset"

In [4]:
dataset_path

PosixPath('military_vehicles_dataset')

In [5]:
import os

def check_data(dir_path):

  for dirpath, dirnames, filenames in os.walk(dir_path):
    print(f"# of directories: {len(dirnames)} and {len(filenames)} in {dirpath}")

In [6]:
check_data(dataset_path)

# of directories: 2 and 0 in military_vehicles_dataset
# of directories: 22 and 0 in military_vehicles_dataset/test
# of directories: 0 and 105 in military_vehicles_dataset/test/Iskander
# of directories: 0 and 21 in military_vehicles_dataset/test/30N6E
# of directories: 0 and 27 in military_vehicles_dataset/test/BMP-1
# of directories: 0 and 59 in military_vehicles_dataset/test/2S19_MSTA
# of directories: 0 and 20 in military_vehicles_dataset/test/T-80
# of directories: 0 and 15 in military_vehicles_dataset/test/T-90
# of directories: 0 and 8 in military_vehicles_dataset/test/T-64
# of directories: 0 and 64 in military_vehicles_dataset/test/BRDM
# of directories: 0 and 87 in military_vehicles_dataset/test/MT_LB
# of directories: 0 and 161 in military_vehicles_dataset/test/BTR-60
# of directories: 0 and 22 in military_vehicles_dataset/test/D-30
# of directories: 0 and 9 in military_vehicles_dataset/test/T-72
# of directories: 0 and 39 in military_vehicles_dataset/test/BTR-70
# of direc

In [7]:
train_dir = dataset_path / "train"
test_dir = dataset_path / "test"

In [8]:
train_dir

PosixPath('military_vehicles_dataset/train')

In [9]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode

In [10]:
manual_transforms = transforms.Compose([
    transforms.Resize(size=(288,288), interpolation=InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(0.3),
    transforms.RandomVerticalFlip(0.2),
    transforms.RandomRotation(degrees=30),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])])

In [11]:
NUM_WORKERS = os.cpu_count()


def create_dataloader(
    train_dir: str,
    test_dir: str,
    transform: transforms.Compose,
    batch_size: int,
    num_workers: int = NUM_WORKERS):

  train_data = datasets.ImageFolder(train_dir, transform = transform)
  test_data = datasets.ImageFolder(test_dir, transform = transform)

  class_names = train_data.classes


  train_dataloader = DataLoader(train_data,
                                batch_size= batch_size,
                                shuffle=True,
                                num_workers=num_workers,
                                pin_memory=True)


  test_dataloader = DataLoader(test_data,
                               batch_size=batch_size,
                               shuffle=True,
                               num_workers=num_workers,
                               pin_memory=False)

  return train_dataloader, test_dataloader, class_names

In [12]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

In [13]:
device

'cuda'

In [14]:
import torchvision

In [15]:
weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT

In [16]:
weights

EfficientNet_B2_Weights.IMAGENET1K_V1

In [17]:
auto_transforms = weights.transforms()
auto_transforms

ImageClassification(
    crop_size=[288]
    resize_size=[288]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

In [18]:
train_dataloader, test_dataloader, class_names = create_dataloader(train_dir=train_dir,
                  test_dir=test_dir,
                  transform = auto_transforms,
                  batch_size=32)

In [19]:
model = torchvision.models.efficientnet_b2(weights=weights).to(device)

In [20]:
model

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [21]:
#!pip install torchinfo

In [22]:
from torchinfo import summary

In [23]:
summary(model=model,
        input_size=(32, 3, 288, 288),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])


Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [32, 3, 288, 288]    [32, 1000]           --                   True
├─Sequential (features)                                      [32, 3, 288, 288]    [32, 1408, 9, 9]     --                   True
│    └─Conv2dNormActivation (0)                              [32, 3, 288, 288]    [32, 32, 144, 144]   --                   True
│    │    └─Conv2d (0)                                       [32, 3, 288, 288]    [32, 32, 144, 144]   864                  True
│    │    └─BatchNorm2d (1)                                  [32, 32, 144, 144]   [32, 32, 144, 144]   64                   True
│    │    └─SiLU (2)                                         [32, 32, 144, 144]   [32, 32, 144, 144]   --                   --
│    └─Sequential (1)                                        [32, 32, 144, 144]   [32, 16, 144

In [24]:
for param in model.features.parameters():
    param.requires_grad = False

In [25]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

output_shape = len(class_names)

model.classifier = torch.nn.Sequential(
    torch.nn.Dropout(0.3, inplace=True),
    torch.nn.Linear(in_features=1408, out_features=output_shape, bias=True).to(device))

In [26]:
summary(model,
        input_size=(32,3,288,288),
        verbose=0,
        col_names=["input_size","output_size","num_params","trainable"],
        col_width=20,
        row_settings=["var_names"])

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [32, 3, 288, 288]    [32, 22]             --                   Partial
├─Sequential (features)                                      [32, 3, 288, 288]    [32, 1408, 9, 9]     --                   False
│    └─Conv2dNormActivation (0)                              [32, 3, 288, 288]    [32, 32, 144, 144]   --                   False
│    │    └─Conv2d (0)                                       [32, 3, 288, 288]    [32, 32, 144, 144]   (864)                False
│    │    └─BatchNorm2d (1)                                  [32, 32, 144, 144]   [32, 32, 144, 144]   (64)                 False
│    │    └─SiLU (2)                                         [32, 32, 144, 144]   [32, 32, 144, 144]   --                   --
│    └─Sequential (1)                                        [32, 32, 144, 144]   [32, 

In [27]:
from torch import nn

In [28]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [29]:
def train_step(
    model: torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    loss_fn: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device
):

  model.train()

  train_loss, train_acc = 0, 0

  for batch, (X, y) in enumerate(dataloader):

    X, y = X.to(device), y.to(device)

    y_pred = model(X)

    loss = loss_fn(y_pred, y)

    train_loss += loss.item()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    y_pred_labels = torch.softmax(y_pred, dim=1).argmax(dim=1)
    train_acc += (y_pred_labels == y).sum().item() / len(y_pred)

  train_loss = train_loss / len(dataloader)
  train_acc = train_acc / len(dataloader)

  return train_loss, train_acc




def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              device: torch.device):

  model.eval()

  test_acc, test_loss = 0, 0

  with torch.inference_mode():

    for batch, (X, y) in enumerate(dataloader):

      X, y = X.to(device), y.to(device)

      test_pred = model(X)

      loss = loss_fn(test_pred, y)
      test_loss += loss.item()

      test_pred_labels = torch.softmax(test_pred, dim=1).argmax(dim=1)
      test_acc += (test_pred_labels == y).sum().item() / len(test_pred)

  test_loss = test_loss / len(dataloader)
  test_acc = test_acc / len(dataloader)

  return test_loss, test_acc




def main_train(model: torch.nn.Module,
               train_dataloader: torch.utils.data.DataLoader,
               test_dataloader: torch.utils.data.DataLoader,
               optimizer: torch.optim.Optimizer,
               device: torch.device,
               loss_fn: torch.nn.Module = nn.CrossEntropyLoss(),
               epochs: int = 10):


  results = {"train_loss": [],
             "train_acc" : [],
             "test_loss": [],
             "test_acc": []}


  for epoch in range(epochs):

    train_loss, train_acc = train_step(model=model,
                                       dataloader=train_dataloader,
                                       loss_fn=loss_fn,
                                       optimizer=optimizer,
                                       device=device)

    test_loss, test_acc = test_step(model=model,
                                    dataloader=test_dataloader,
                                    loss_fn=loss_fn,
                                    device=device)

    print(
        f"Epoch: {epoch + 1}  |"
        f"train_loss: {train_loss:.4f}  |"
        f"train_acc: {train_acc:.4f}  |"
        f"test_loss: {test_loss:.4f}  |"
        f"test_acc: {test_acc:.4}"
        )


    results["train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
    results["train_acc"].append(train_acc.item() if isinstance(train_acc, torch.Tensor) else train_acc)
    results["test_loss"].append(test_loss.item() if isinstance(test_loss, torch.Tensor) else test_loss)
    results["test_acc"].append(test_acc.item() if isinstance(test_acc, torch.Tensor) else test_acc)

  return results

In [30]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

from timeit import default_timer as timer
start_time = timer()

results = main_train(model=model,
                train_dataloader=train_dataloader,
                test_dataloader=test_dataloader,
                optimizer=optimizer,
                loss_fn=loss_fn,
                epochs=30,
                device=device)

Epoch: 1  |train_loss: 2.4540  |train_acc: 0.2856  |test_loss: 2.2571  |test_acc: 0.3998
Epoch: 2  |train_loss: 1.9486  |train_acc: 0.4424  |test_loss: 2.0917  |test_acc: 0.4167
Epoch: 3  |train_loss: 1.7529  |train_acc: 0.4735  |test_loss: 1.9064  |test_acc: 0.4736
Epoch: 4  |train_loss: 1.6313  |train_acc: 0.5138  |test_loss: 1.8855  |test_acc: 0.4671
Epoch: 5  |train_loss: 1.5502  |train_acc: 0.5257  |test_loss: 1.8335  |test_acc: 0.4784
Epoch: 6  |train_loss: 1.4826  |train_acc: 0.5511  |test_loss: 1.8239  |test_acc: 0.48
Epoch: 7  |train_loss: 1.4341  |train_acc: 0.5617  |test_loss: 1.8425  |test_acc: 0.4615
Epoch: 8  |train_loss: 1.3999  |train_acc: 0.5705  |test_loss: 1.8119  |test_acc: 0.484
Epoch: 9  |train_loss: 1.3571  |train_acc: 0.5798  |test_loss: 1.8017  |test_acc: 0.484
Epoch: 10  |train_loss: 1.3553  |train_acc: 0.5733  |test_loss: 1.7679  |test_acc: 0.5008
Epoch: 11  |train_loss: 1.3100  |train_acc: 0.5901  |test_loss: 1.7655  |test_acc: 0.496
Epoch: 12  |train_loss: 